# Sanity checks: published and independent Yamada references

This benchmark entry executes the canonical `User_guide/Sanity_checks.ipynb` from the current checkout. It is intentionally a thin runner so the published-reference checks have **one source of truth** and cannot drift between two copied notebooks.

The canonical notebook checks closed-form graph families, bridge/isthmus and one-point-union identities, the published planar $K_4$ value, an independent Negami subset calculation, agreement of the public Yamada backends, and a crossing-containing spatial graph with its mirror.

A successful run means every assertion in the canonical sanity notebook passed using the `knotted_graph` source from this checkout.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'knotted_graph').exists():
    raise RuntimeError('Run from inside the KnottedGraph checkout.')
SRC = ROOT / 'src'
sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f'Stale knotted_graph import: {kg_path}')

branch = subprocess.check_output(['git','rev-parse','--abbrev-ref','HEAD'], cwd=ROOT, text=True).strip()
commit = subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()
print('branch =', branch)
print('commit =', commit)
print('knotted_graph =', kg_path)


## Execute every code cell from the canonical sanity notebook

In [ ]:
canonical = ROOT / 'User_guide' / 'Sanity_checks.ipynb'
if not canonical.exists():
    raise FileNotFoundError(canonical)

with canonical.open(encoding='utf-8') as handle:
    notebook = json.load(handle)

namespace = {'__name__': '__sanity_benchmark__'}
executed = 0
for cell_index, cell in enumerate(notebook.get('cells', [])):
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    if not source.strip():
        continue
    try:
        exec(compile(source, f'{canonical.name}:cell-{cell_index}', 'exec'), namespace, namespace)
    except Exception as exc:
        raise RuntimeError(
            f'Canonical sanity check failed in code cell {cell_index}: {type(exc).__name__}: {exc}'
        ) from exc
    executed += 1

print(f'PASS: executed {executed} canonical sanity-check code cells with no failed assertions.')


## Acceptance criterion

There is no performance timing in this notebook. It is the correctness gate that must pass before benchmark speedups are accepted.